In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Working directory: /home/smallyan/eval_agent


CUDA available: True
GPU: NVIDIA A40


# No-Execution Code Evaluation - Erasing-LLM

## Evaluation Mode
This evaluation is conducted in **No-Execution Code Evaluation** mode.

**Repository Path:** `/net/scratch2/smallyan/erasing-llm_eval`

### Source Files Used:
1. **Plan File:** `/net/scratch2/smallyan/erasing-llm_eval/plan.md`
2. **CodeWalk File:** `/net/scratch2/smallyan/erasing-llm_eval/CodeWalkthrough.md`

### Code Files to Evaluate:
1. `trainscripts/erase.py` - Main training script for ELM method
2. `trainscripts/prepare_consistency_data.py` - Data preparation script
3. `utils/metrics.py` - Evaluation metrics functions
4. `utils/lora.py` - LoRA network implementation
5. `notebooks/inference.ipynb` - Inference notebook

---

## Project Goal Summary

From the Plan file, the ELM (Erasure of Language Memory) method aims to:
1. Use introspective classification by leveraging implicit model probabilities with two context prompts (expert vs novice)
2. Combine three loss terms: Lerase, Lretain, and Lfluency
3. Apply low-rank adapters (LoRA) on early model layers (layers 4-7)
4. Train on erase datasets and retain datasets with expert/novice context prompts

In [2]:
import pandas as pd

# Complete evaluation data for all 38 code blocks
evaluation_data = [
    # trainscripts/erase.py (10 blocks)
    {'File': 'trainscripts/erase.py', 'Block': 'Imports and setup (lines 1-33)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': ''},
    {'File': 'trainscripts/erase.py', 'Block': 'get_edit_vector() (lines 34-105)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': 'Implements the core ELM probability adjustment: original + eta*(expert - novice), consistent with plan'},
    {'File': 'trainscripts/erase.py', 'Block': 'ELMLogits class (lines 113-151)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': 'LogitsProcessor implementation for ELM-guided generation'},
    {'File': 'trainscripts/erase.py', 'Block': 'generate() (lines 152-175)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': 'Text generation with ELM logits processor'},
    {'File': 'trainscripts/erase.py', 'Block': 'prepare_prompts() (lines 177-264)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': 'Loads WMDP bio/cyber/HP datasets as described in plan'},
    {'File': 'trainscripts/erase.py', 'Block': 'moving_average() (lines 265-268)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'Y', 'Error_Notes': 'Function defined but not used anywhere in the codebase'},
    {'File': 'trainscripts/erase.py', 'Block': 'Prompt templates (lines 272-319)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': 'Expert/novice prompt templates as per plan methodology'},
    {'File': 'trainscripts/erase.py', 'Block': 'train_elm() (lines 323-660)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': 'Main training loop implementing Lerase, Lretain, Lfluency loss terms with LoRA on configurable layers'},
    {'File': 'trainscripts/erase.py', 'Block': 'argparse setup (lines 662-887)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': 'Command-line argument parsing with sensible defaults (layers 4-8, rank 256, eta 1000)'},
    {'File': 'trainscripts/erase.py', 'Block': '__main__ execution (lines 889-934)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': 'Training and evaluation flow with WMDP and MMLU evaluation'},
    
    # trainscripts/prepare_consistency_data.py (6 blocks)
    {'File': 'trainscripts/prepare_consistency_data.py', 'Block': 'Imports (lines 1-31)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': ''},
    {'File': 'trainscripts/prepare_consistency_data.py', 'Block': 'ELMLogits class (lines 32-70)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'Y', 'Irrelevant': 'N', 'Error_Notes': 'Duplicates ELMLogits class from erase.py (identical implementation)'},
    {'File': 'trainscripts/prepare_consistency_data.py', 'Block': 'generate() (lines 71-92)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'Y', 'Irrelevant': 'N', 'Error_Notes': 'Duplicates generate() from erase.py with minor differences'},
    {'File': 'trainscripts/prepare_consistency_data.py', 'Block': 'prepare_prompts() (lines 94-181)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'Y', 'Irrelevant': 'N', 'Error_Notes': 'Duplicates prepare_prompts() from erase.py'},
    {'File': 'trainscripts/prepare_consistency_data.py', 'Block': 'Prompt templates (lines 184-231)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'Y', 'Irrelevant': 'N', 'Error_Notes': 'Duplicates prompt templates from erase.py'},
    {'File': 'trainscripts/prepare_consistency_data.py', 'Block': '__main__ (lines 234-394)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': 'Pre-generates consistency data for faster training'},
    
    # utils/metrics.py (11 blocks)
    {'File': 'utils/metrics.py', 'Block': 'Imports and ans_map (lines 1-21)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': ''},
    {'File': 'utils/metrics.py', 'Block': 'prepare_data() (lines 23-44)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': 'Batches MCQ data for evaluation'},
    {'File': 'utils/metrics.py', 'Block': 'prepare_data_wmdp() (lines 47-69)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': 'WMDP-specific MCQ batching'},
    {'File': 'utils/metrics.py', 'Block': 'prepare_data_hp() (lines 71-91)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': 'Harry Potter MCQ batching'},
    {'File': 'utils/metrics.py', 'Block': 'prepare_data_truthfulqa() (lines 93-111)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': 'Binary choice format for TruthfulQA'},
    {'File': 'utils/metrics.py', 'Block': 'get_accuracy() (lines 113-136)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': '4-choice MCQ accuracy using logits at A/B/C/D tokens'},
    {'File': 'utils/metrics.py', 'Block': 'get_accuracy_binary() (lines 138-160)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': 'Binary choice accuracy for TruthfulQA/HP dual'},
    {'File': 'utils/metrics.py', 'Block': 'get_wmdp_accuracy() (lines 162-188)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': 'Evaluates WMDP bio and cyber benchmarks'},
    {'File': 'utils/metrics.py', 'Block': 'get_mmlu_accuracy() (lines 190-209)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': 'MMLU evaluation across all subjects'},
    {'File': 'utils/metrics.py', 'Block': 'get_hp_accuracy() (lines 212-228)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': 'Harry Potter MCQ evaluation'},
    {'File': 'utils/metrics.py', 'Block': 'get_truthfulqa() (lines 229-242)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': 'TruthfulQA binary evaluation'},
    
    # utils/lora.py (6 blocks)
    {'File': 'utils/lora.py', 'Block': 'Imports and constants (lines 1-22)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': ''},
    {'File': 'utils/lora.py', 'Block': 'LoRAModule class (lines 25-71)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'Y', 'Error_Notes': 'Custom LoRA implementation; not used because erase.py uses peft library LoRA instead'},
    {'File': 'utils/lora.py', 'Block': 'LoRANetwork class (lines 74-167)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'Y', 'Error_Notes': 'Custom LoRA network wrapper; not used because erase.py uses peft library instead'},
    {'File': 'utils/lora.py', 'Block': 'prepare_optimizer_params() (lines 168-177)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'Y', 'Error_Notes': 'Part of unused custom LoRA implementation'},
    {'File': 'utils/lora.py', 'Block': 'save_weights() (lines 179-196)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'Y', 'Error_Notes': 'Part of unused custom LoRA implementation'},
    {'File': 'utils/lora.py', 'Block': 'Context manager (lines 197-206)', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'Y', 'Error_Notes': 'Part of unused custom LoRA implementation; __enter__/__exit__ for LoRA activation'},
    
    # notebooks/inference.ipynb (5 cells)
    {'File': 'notebooks/inference.ipynb', 'Block': 'Cell 1: Imports', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': ''},
    {'File': 'notebooks/inference.ipynb', 'Block': 'Cell 2: Model loading', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': 'Loads model and tokenizer with proper config'},
    {'File': 'notebooks/inference.ipynb', 'Block': 'Cell 3: load_peft()', 'Runnable': 'N', 'Correct_Implementation': 'N', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': 'Path "../lora_models/my_elm/checkpoint-final/" is hardcoded and will not exist without prior training; model.unload() exception handling is fragile'},
    {'File': 'notebooks/inference.ipynb', 'Block': 'Cell 4: generate_text()', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': 'Text generation helper function'},
    {'File': 'notebooks/inference.ipynb', 'Block': 'Cell 5: Generation example', 'Runnable': 'Y', 'Correct_Implementation': 'Y', 'Redundant': 'N', 'Irrelevant': 'N', 'Error_Notes': 'Tests bioweapon prompt to demonstrate erasure'},
]

df = pd.DataFrame(evaluation_data)
print(f"Total blocks evaluated: {len(df)}")

Total blocks evaluated: 38


## Block-Level Evaluation Table

In [3]:
# Display the full evaluation table
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_rows', None)

print("=" * 140)
print("BLOCK-LEVEL EVALUATION TABLE")
print("=" * 140)
print(df.to_string(index=True))
print("\n")
print(f"Total blocks evaluated: {len(df)}")

BLOCK-LEVEL EVALUATION TABLE
                                        File                                       Block Runnable Correct_Implementation Redundant Irrelevant                                                                                                                                          Error_Notes
0                      trainscripts/erase.py              Imports and setup (lines 1-33)        Y                      Y         N          N                                                                                                                                                     
1                      trainscripts/erase.py            get_edit_vector() (lines 34-105)        Y                      Y         N          N                                               Implements the core ELM probability adjustment: original + eta*(expert - novice), consistent with plan
2                      trainscripts/erase.py             ELMLogits class (lines 113-151)        Y 

## Quantitative Metrics

In [4]:
# =========================================
# COMPUTE QUANTITATIVE METRICS
# =========================================

total_blocks = len(df)

# Runnable%
runnable_count = (df['Runnable'] == 'Y').sum()
runnable_pct = (runnable_count / total_blocks) * 100

# Incorrect% (Correct_Implementation = N)
incorrect_count = (df['Correct_Implementation'] == 'N').sum()
incorrect_pct = (incorrect_count / total_blocks) * 100

# Redundant%
redundant_count = (df['Redundant'] == 'Y').sum()
redundant_pct = (redundant_count / total_blocks) * 100

# Irrelevant%
irrelevant_count = (df['Irrelevant'] == 'Y').sum()
irrelevant_pct = (irrelevant_count / total_blocks) * 100

# Failed blocks (either Runnable=N or Correct_Implementation=N)
failed_blocks = ((df['Runnable'] == 'N') | (df['Correct_Implementation'] == 'N')).sum()

# Blocks with identified corrections (the only failing block, Cell 3 in inference.ipynb, has an identified fix)
# Fix: Make the path a configurable parameter and use proper error handling
blocks_with_corrections = 1

correction_rate_pct = (blocks_with_corrections / failed_blocks) * 100 if failed_blocks > 0 else 100.0

print("=" * 80)
print("QUANTITATIVE METRICS")
print("=" * 80)
print(f"Total blocks evaluated: {total_blocks}")
print()
print(f"Runnable%:              {runnable_pct:.2f}% ({runnable_count}/{total_blocks})")
print(f"Incorrect%:             {incorrect_pct:.2f}% ({incorrect_count}/{total_blocks})")
print(f"Redundant%:             {redundant_pct:.2f}% ({redundant_count}/{total_blocks})")
print(f"Irrelevant%:            {irrelevant_pct:.2f}% ({irrelevant_count}/{total_blocks})")
print(f"Correction-Rate%:       {correction_rate_pct:.2f}% ({blocks_with_corrections}/{failed_blocks} failing blocks have identified fixes)")
print()
print("=" * 80)
print("NOTE: The only failing block (Cell 3 in inference.ipynb) has an identified fix:")
print("  - Make peft_path a configurable parameter instead of hardcoded path")
print("  - Use proper existence check instead of try/except for model.unload()")
print("=" * 80)

# Store metrics for later
metrics = {
    'total_blocks': total_blocks,
    'runnable_count': runnable_count,
    'incorrect_count': incorrect_count,
    'redundant_count': redundant_count,
    'irrelevant_count': irrelevant_count,
    'runnable_pct': runnable_pct,
    'incorrect_pct': incorrect_pct,
    'redundant_pct': redundant_pct,
    'irrelevant_pct': irrelevant_pct,
    'correction_rate_pct': correction_rate_pct
}

QUANTITATIVE METRICS
Total blocks evaluated: 38

Runnable%:              97.37% (37/38)
Incorrect%:             2.63% (1/38)
Redundant%:             10.53% (4/38)
Irrelevant%:            15.79% (6/38)
Correction-Rate%:       100.00% (1/1 failing blocks have identified fixes)

NOTE: The only failing block (Cell 3 in inference.ipynb) has an identified fix:
  - Make peft_path a configurable parameter instead of hardcoded path
  - Use proper existence check instead of try/except for model.unload()


## Binary Checklist Summary (C1-C4)

In [5]:
# =========================================
# BINARY CHECKLIST SUMMARY (C1-C4)
# =========================================

# C1: All core analysis code is runnable
c1_pass = (df['Runnable'] == 'N').sum() == 0
c1_status = "PASS" if c1_pass else "FAIL"
c1_reason = "1 block has Runnable=N (notebooks/inference.ipynb Cell 3: hardcoded path that won't exist)" if not c1_pass else "All blocks are runnable"

# C2: All implementations are correct
c2_pass = (df['Correct_Implementation'] == 'N').sum() == 0
c2_status = "PASS" if c2_pass else "FAIL"
c2_reason = "1 block has Correct_Implementation=N (notebooks/inference.ipynb Cell 3: hardcoded path issue)" if not c2_pass else "All implementations are correct"

# C3: No redundant code
c3_pass = (df['Redundant'] == 'Y').sum() == 0
c3_status = "PASS" if c3_pass else "FAIL"
c3_reason = f"4 blocks have Redundant=Y (prepare_consistency_data.py duplicates code from erase.py)" if not c3_pass else "No redundant code"

# C4: No irrelevant code
c4_pass = (df['Irrelevant'] == 'Y').sum() == 0
c4_status = "PASS" if c4_pass else "FAIL"
c4_reason = f"6 blocks have Irrelevant=Y (utils/lora.py custom LoRA not used, moving_average() unused)" if not c4_pass else "No irrelevant code"

# Create checklist table
checklist_data = [
    {'Checklist Item': 'C1: All core analysis code is runnable', 'Condition': 'No block has Runnable=N', 'PASS/FAIL': c1_status},
    {'Checklist Item': 'C2: All implementations are correct', 'Condition': 'No block has Correct_Implementation=N', 'PASS/FAIL': c2_status},
    {'Checklist Item': 'C3: No redundant code', 'Condition': 'No block has Redundant=Y', 'PASS/FAIL': c3_status},
    {'Checklist Item': 'C4: No irrelevant code', 'Condition': 'No block has Irrelevant=Y', 'PASS/FAIL': c4_status},
]

checklist_df = pd.DataFrame(checklist_data)

print("=" * 100)
print("BINARY CHECKLIST SUMMARY")
print("=" * 100)
print(checklist_df.to_string(index=False))
print()
print("=" * 100)
print("DETAILED RATIONALE:")
print("=" * 100)
print(f"C1: {c1_status} - {c1_reason}")
print(f"C2: {c2_status} - {c2_reason}")
print(f"C3: {c3_status} - {c3_reason}")
print(f"C4: {c4_status} - {c4_reason}")
print("=" * 100)

# Store for JSON
checklist = {
    'C1_All_Runnable': c1_status,
    'C2_All_Correct': c2_status,
    'C3_No_Redundant': c3_status,
    'C4_No_Irrelevant': c4_status
}

rationale = {
    'C1_All_Runnable': c1_reason,
    'C2_All_Correct': c2_reason,
    'C3_No_Redundant': c3_reason,
    'C4_No_Irrelevant': c4_reason
}

BINARY CHECKLIST SUMMARY
                        Checklist Item                             Condition PASS/FAIL
C1: All core analysis code is runnable               No block has Runnable=N      FAIL
   C2: All implementations are correct No block has Correct_Implementation=N      FAIL
                 C3: No redundant code              No block has Redundant=Y      FAIL
                C4: No irrelevant code             No block has Irrelevant=Y      FAIL

DETAILED RATIONALE:
C1: FAIL - 1 block has Runnable=N (notebooks/inference.ipynb Cell 3: hardcoded path that won't exist)
C2: FAIL - 1 block has Correct_Implementation=N (notebooks/inference.ipynb Cell 3: hardcoded path issue)
C3: FAIL - 4 blocks have Redundant=Y (prepare_consistency_data.py duplicates code from erase.py)
C4: FAIL - 6 blocks have Irrelevant=Y (utils/lora.py custom LoRA not used, moving_average() unused)


## Summary

### Overall Assessment

The ELM (Erasure of Language Memory) codebase implements the methodology described in the plan:
- **Core ELM algorithm** in `get_edit_vector()` correctly implements the probability adjustment formula
- **Three loss terms** (Lerase, Lretain, Lfluency) are properly implemented in `train_elm()`
- **LoRA integration** uses the peft library for low-rank adapter training on configurable layers
- **Evaluation metrics** cover WMDP, MMLU, Harry Potter, and TruthfulQA benchmarks

### Issues Found

1. **Runnability Issue (1 block)**: `notebooks/inference.ipynb` Cell 3 has a hardcoded path that won't exist without prior training
   - **Fix**: Make `peft_path` a configurable parameter or check for file existence before loading

2. **Redundant Code (4 blocks)**: `prepare_consistency_data.py` duplicates several components from `erase.py`:
   - ELMLogits class
   - generate() function
   - prepare_prompts() function
   - Prompt templates
   - **Fix**: Factor these into a shared module

3. **Irrelevant Code (6 blocks)**:
   - `utils/lora.py` contains a complete custom LoRA implementation that is never used (peft is used instead)
   - `moving_average()` function in erase.py is defined but never called
   - **Fix**: Remove unused code or clearly document it as legacy/alternative implementation

### Final Metrics Summary

| Metric | Value |
|--------|-------|
| Runnable% | 97.37% |
| Incorrect% | 2.63% |
| Redundant% | 10.53% |
| Irrelevant% | 15.79% |
| Correction-Rate% | 100.00% |

In [6]:
# =========================================
# SAVE JSON SUMMARY
# =========================================
import json
import os

# Create output directory
output_dir = '/net/scratch2/smallyan/erasing-llm_eval/no_exe_evaluation'
os.makedirs(output_dir, exist_ok=True)

# Create JSON summary
json_summary = {
    "Runnable_Percentage": round(metrics['runnable_pct'], 2),
    "Incorrect_Percentage": round(metrics['incorrect_pct'], 2),
    "Redundant_Percentage": round(metrics['redundant_pct'], 2),
    "Irrelevant_Percentage": round(metrics['irrelevant_pct'], 2),
    "Correction_Rate_Percentage": round(metrics['correction_rate_pct'], 2),
    
    "Issues": {
        "Runnable_Issues_Exist": metrics['runnable_count'] < metrics['total_blocks'],
        "Output_Mismatch_Exists": False,  # No output to compare in no-execution mode
        "Incorrect_Exists": metrics['incorrect_count'] > 0,
        "Redundant_Exists": metrics['redundant_count'] > 0,
        "Irrelevant_Exists": metrics['irrelevant_count'] > 0
    },
    
    "Checklist": checklist,
    
    "Rationale": rationale
}

# Save JSON
json_path = os.path.join(output_dir, 'code_critic_summary.json')
with open(json_path, 'w') as f:
    json.dump(json_summary, f, indent=2)

print(f"JSON summary saved to: {json_path}")
print()
print("JSON Contents:")
print(json.dumps(json_summary, indent=2))

TypeError: Object of type bool is not JSON serializable

In [7]:
# Fix the numpy bool issue and save JSON
import json
import os
import numpy as np

# Convert numpy booleans to Python booleans
def convert_to_python_types(obj):
    if isinstance(obj, dict):
        return {k: convert_to_python_types(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_python_types(item) for item in obj]
    elif isinstance(obj, (np.bool_, np.integer)):
        return bool(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    else:
        return obj

# Create output directory
output_dir = '/net/scratch2/smallyan/erasing-llm_eval/no_exe_evaluation'
os.makedirs(output_dir, exist_ok=True)

# Create JSON summary with Python native types
json_summary = {
    "Runnable_Percentage": float(round(metrics['runnable_pct'], 2)),
    "Incorrect_Percentage": float(round(metrics['incorrect_pct'], 2)),
    "Redundant_Percentage": float(round(metrics['redundant_pct'], 2)),
    "Irrelevant_Percentage": float(round(metrics['irrelevant_pct'], 2)),
    "Correction_Rate_Percentage": float(round(metrics['correction_rate_pct'], 2)),
    
    "Issues": {
        "Runnable_Issues_Exist": bool(int(metrics['runnable_count']) < int(metrics['total_blocks'])),
        "Output_Mismatch_Exists": False,
        "Incorrect_Exists": bool(int(metrics['incorrect_count']) > 0),
        "Redundant_Exists": bool(int(metrics['redundant_count']) > 0),
        "Irrelevant_Exists": bool(int(metrics['irrelevant_count']) > 0)
    },
    
    "Checklist": {
        "C1_All_Runnable": str(checklist['C1_All_Runnable']),
        "C2_All_Correct": str(checklist['C2_All_Correct']),
        "C3_No_Redundant": str(checklist['C3_No_Redundant']),
        "C4_No_Irrelevant": str(checklist['C4_No_Irrelevant'])
    },
    
    "Rationale": {
        "C1_All_Runnable": str(rationale['C1_All_Runnable']),
        "C2_All_Correct": str(rationale['C2_All_Correct']),
        "C3_No_Redundant": str(rationale['C3_No_Redundant']),
        "C4_No_Irrelevant": str(rationale['C4_No_Irrelevant'])
    }
}

# Save JSON
json_path = os.path.join(output_dir, 'code_critic_summary.json')
with open(json_path, 'w') as f:
    json.dump(json_summary, f, indent=2)

print(f"JSON summary saved to: {json_path}")
print()
print("JSON Contents:")
print(json.dumps(json_summary, indent=2))

JSON summary saved to: /net/scratch2/smallyan/erasing-llm_eval/no_exe_evaluation/code_critic_summary.json

JSON Contents:
{
  "Runnable_Percentage": 97.37,
  "Incorrect_Percentage": 2.63,
  "Redundant_Percentage": 10.53,
  "Irrelevant_Percentage": 15.79,
  "Correction_Rate_Percentage": 100.0,
  "Issues": {
    "Runnable_Issues_Exist": true,
    "Output_Mismatch_Exists": false,
    "Incorrect_Exists": true,
    "Redundant_Exists": true,
    "Irrelevant_Exists": true
  },
  "Checklist": {
    "C1_All_Runnable": "FAIL",
    "C2_All_Correct": "FAIL",
    "C3_No_Redundant": "FAIL",
    "C4_No_Irrelevant": "FAIL"
  },
  "Rationale": {
    "C1_All_Runnable": "1 block has Runnable=N (notebooks/inference.ipynb Cell 3: hardcoded path that won't exist)",
    "C2_All_Correct": "1 block has Correct_Implementation=N (notebooks/inference.ipynb Cell 3: hardcoded path issue)",
    "C3_No_Redundant": "4 blocks have Redundant=Y (prepare_consistency_data.py duplicates code from erase.py)",
    "C4_No_Irre

In [8]:
# Save the evaluation table to CSV for reference
csv_path = os.path.join(output_dir, 'block_evaluation_table.csv')
df.to_csv(csv_path, index=False)
print(f"Block evaluation table saved to: {csv_path}")

# Verify files exist
print("\nVerifying output files:")
for f in [json_path, csv_path]:
    if os.path.exists(f):
        print(f"  ✓ {f} ({os.path.getsize(f)} bytes)")
    else:
        print(f"  ✗ {f} NOT FOUND")

Block evaluation table saved to: /net/scratch2/smallyan/erasing-llm_eval/no_exe_evaluation/block_evaluation_table.csv

Verifying output files:
  ✓ /net/scratch2/smallyan/erasing-llm_eval/no_exe_evaluation/code_critic_summary.json (983 bytes)
  ✓ /net/scratch2/smallyan/erasing-llm_eval/no_exe_evaluation/block_evaluation_table.csv (4269 bytes)


In [9]:
# Copy the notebook to the required output location
import shutil
import time

# Wait briefly to ensure notebook is saved
time.sleep(1)

# Source notebook path
source_notebook = '/home/smallyan/eval_agent/notebooks/2026-01-15-11-19_CodeCriticEval.ipynb'

# Destination path
dest_notebook = '/net/scratch2/smallyan/erasing-llm_eval/no_exe_evaluation/code_critic_evaluation.ipynb'

# Copy the notebook
shutil.copy2(source_notebook, dest_notebook)

print(f"Notebook copied to: {dest_notebook}")

# Verify all output files exist
print("\nFinal verification of all output files:")
output_files = [
    '/net/scratch2/smallyan/erasing-llm_eval/no_exe_evaluation/code_critic_evaluation.ipynb',
    '/net/scratch2/smallyan/erasing-llm_eval/no_exe_evaluation/code_critic_summary.json'
]

for f in output_files:
    if os.path.exists(f):
        print(f"  ✓ {f} ({os.path.getsize(f)} bytes)")
    else:
        print(f"  ✗ {f} NOT FOUND")

Notebook copied to: /net/scratch2/smallyan/erasing-llm_eval/no_exe_evaluation/code_critic_evaluation.ipynb

Final verification of all output files:
  ✓ /net/scratch2/smallyan/erasing-llm_eval/no_exe_evaluation/code_critic_evaluation.ipynb (54398 bytes)
  ✓ /net/scratch2/smallyan/erasing-llm_eval/no_exe_evaluation/code_critic_summary.json (983 bytes)


In [10]:
# Final update to copy notebook with all content
import time
time.sleep(2)

shutil.copy2(source_notebook, dest_notebook)

print("=" * 80)
print("EVALUATION COMPLETE")
print("=" * 80)
print()
print("Output files created:")
print(f"  1. Notebook: {dest_notebook}")
print(f"  2. JSON Summary: /net/scratch2/smallyan/erasing-llm_eval/no_exe_evaluation/code_critic_summary.json")
print()
print("=" * 80)
print("SUMMARY OF RESULTS")
print("=" * 80)
print(f"Total blocks evaluated: 38")
print(f"Runnable%:              97.37% (37/38)")
print(f"Incorrect%:             2.63% (1/38)")
print(f"Redundant%:             10.53% (4/38)")
print(f"Irrelevant%:            15.79% (6/38)")
print(f"Correction-Rate%:       100.00%")
print()
print("Checklist Results:")
print(f"  C1 (All Runnable):      FAIL (1 block has hardcoded path)")
print(f"  C2 (All Correct):       FAIL (1 block has implementation issue)")
print(f"  C3 (No Redundant):      FAIL (4 duplicate blocks in prepare_consistency_data.py)")
print(f"  C4 (No Irrelevant):     FAIL (6 unused blocks: custom LoRA, moving_average)")
print("=" * 80)

EVALUATION COMPLETE

Output files created:
  1. Notebook: /net/scratch2/smallyan/erasing-llm_eval/no_exe_evaluation/code_critic_evaluation.ipynb
  2. JSON Summary: /net/scratch2/smallyan/erasing-llm_eval/no_exe_evaluation/code_critic_summary.json

SUMMARY OF RESULTS
Total blocks evaluated: 38
Runnable%:              97.37% (37/38)
Incorrect%:             2.63% (1/38)
Redundant%:             10.53% (4/38)
Irrelevant%:            15.79% (6/38)
Correction-Rate%:       100.00%

Checklist Results:
  C1 (All Runnable):      FAIL (1 block has hardcoded path)
  C2 (All Correct):       FAIL (1 block has implementation issue)
  C3 (No Redundant):      FAIL (4 duplicate blocks in prepare_consistency_data.py)
  C4 (No Irrelevant):     FAIL (6 unused blocks: custom LoRA, moving_average)
